In [10]:
import numpy as np
import pandas as pd
import gc
import os
import json
import nltk
nltk.download('punkt')  # Download the sentence tokenizer
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/zixuanwu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/zixuanwu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
import json
f= open("filtered_metadata.json")
filtered_data = json.load(f)

In [7]:
df = pd.DataFrame(filtered_data)

df["update_date"] = pd.to_datetime(df["update_date"])

In [16]:
import pandas as pd

# Create initial DataFrame
df["doc_id"] = df["id"]
theory_cats = {"hep-th", "math-ph", "gr-qc", "cond-mat.th", "nucl-th"}
computation_cats = {"physics.comp-ph", "cs.NA"}
ml_cats = {"physics.data-an", "cs.LG", "astro-ph.IM"}
applied_cats = {"physics.app-ph", "physics.ins-det", "cond-mat.mtrl-sci", "astro-ph.EP"}
# Step 1: Extract (doc_id, stat_category) pairs
rows = []
for entry in filtered_data:
    doc_id = entry['id']
    abstract = entry['abstract']
    categories = [
        cat for cat in entry["categories"].split()
        if cat in theory_cats or cat in applied_cats or cat in ml_cats or cat in computation_cats
    ]
    if len(categories) == 1:  # Keep only if there's exactly one stat category
        rows.append({
            "doc_id": doc_id,
            "abstract": abstract,
            "physics_category": categories[0],
            "update_date": entry["update_date"]
        })

# Step 2: Create DataFrame with unique (abstract, label) pairs
df_unique = pd.DataFrame(rows).drop_duplicates(subset=["doc_id", "physics_category"])


In [21]:
df_unique.physics_category.unique()

array(['cs.LG', 'physics.data-an', 'physics.comp-ph', 'math-ph', 'cs.NA',
       'cond-mat.mtrl-sci', 'astro-ph.IM', 'nucl-th', 'physics.ins-det',
       'hep-th', 'astro-ph.EP', 'physics.app-ph', 'gr-qc'], dtype=object)

In [12]:
df_unique["physics_category"] = df_unique["physics_category"].replace(
    {**{c: "theory" for c in theory_cats},
     **{c: "application" for c in applied_cats},
     **{c: "computation" for c in computation_cats},
     **{c: "machine_learning" for c in ml_cats}}
)

In [13]:
import pandas as pd

# Column: physics_category

# Find smallest class size
min_count = df_unique["physics_category"].value_counts().min()

# Sample min_count from each category
df_balanced = (
    df_unique.groupby("physics_category", group_keys=False)
      .apply(lambda x: x.sample(min_count, random_state=42))
)

print(df_balanced["physics_category"].value_counts())

physics_category
application         41
computation         41
machine_learning    41
theory              41
Name: count, dtype: int64


In [14]:
df_balanced

,doc_id,abstract,physics_category,update_date
4916,2102.08455,We present an empirical model for nitric oxi...,application,2021-02-18
2741,1906.08506,This paper presents a proof-of-concept demon...,application,2019-12-04
1371,1804.06913,Recent results at the Large Hadron Collider ...,application,2020-06-18
5126,2106.04809,Fractured metal fragments with rough and irr...,application,2024-09-10
783,1609.07407,Conventional LIDAR systems require hundreds ...,application,2019-11-13
...,...,...,...,...
537,1504.01281,We consider four nontrivial ensembles involv...,theory,2015-09-16
713,1605.06459,We detect a certain pattern of behavior of s...,theory,2016-12-30
91,1012.1234,"We calculate the `one-point function', meani...",theory,2015-03-17
1650,1808.05977,"Background: The ""proton radius puzzle"" refer...",theory,2019-05-22


In [20]:
import pandas as pd

df_balanced.to_csv("abstracts_physics.csv", index = False)
